# 03 — Train and score

**Depends on:** notebooks 00–02. **Produces:** `models/*.joblib`,
`scores.parquet`, `metrics.csv`.

## What this stage is for

Fitting anomaly detectors on the clean non-fraud population, scoring a mixed
population, and measuring how well the score ranks the known fraud.

## The premise, stated plainly

There are no fraud labels at training time — a few thousand confirmed cases
against millions of customers is not a training set, it is a test set. So: fit a
model on customers believed innocent, and treat "does not fit the model" as
"suspicious". The confirmed fraud is used **only** to measure whether that
worked.

This is why decontamination in notebook 00 mattered so much. Every fraud customer
left in the training set was teaching the model that fraud is normal.

## Imputation: the part that is actually domain knowledge

Three rules, applied by column name:

- **Counts and flags → 0.** A null count means the event never happened. The
  median of the customers it *did* happen to is a number nobody had.
- **Recency → a large sentinel.** A null `days_since_last` means it has never
  happened. "Never" is further away than any observed value, so fill with
  something larger than the maximum, not with the median. Filling recency with
  the median tells the model this customer changed their SIM a typical amount of
  time ago, which is the opposite of the truth.
- **Entropy, spread, ratios → 0.** Undefined when there is nothing to spread
  over. Zero means "no spread", which is right.

Everything else gets the median. This is `DomainImputer` in the package, written
out longhand.

## Scaling

Every model here is distance-based or density-based, so scale matters
enormously: an unscaled `days_since_first_seen` in the thousands would drown a
count in the single digits. `RobustScaler` is the default because the
distributions are heavy-tailed and `StandardScaler` lets one outlier set the
scale for the whole column.

In [ ]:
import json
import os
import pathlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Every notebook in this track reads and writes here. Nothing else is shared:
# no imports between notebooks, no global state, no kernel that has to stay
# alive. That is what "standalone" means -- you can run notebook 03 tomorrow on
# a different machine as long as this directory came with you.
RUN_DIR = pathlib.Path(os.environ.get("TS05_RUN_DIR", "./_run")).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260601
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print("run directory:", RUN_DIR)
print("contents     :", sorted(p.name for p in RUN_DIR.iterdir()) or "(empty)")

In [ ]:
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
assert "02" in manifest["stages_complete"], "Run 02-select-features.ipynb first."

selection = json.loads((RUN_DIR / "selected_features.json").read_text())

training = pd.read_parquet(manifest["paths"]["training_nonfraud"])
testing = pd.read_parquet(manifest["paths"]["testing_nonfraud"])
fraud = pd.read_parquet(manifest["paths"]["fraud"])
fraud_labels = pd.read_parquet(manifest["paths"]["fraud_labels"])

print("training :", training.shape)
print("testing  :", testing.shape)
print("fraud    :", fraud.shape)
print("available feature lists:", sorted(selection["selected"], key=int))

In [ ]:
TOP_N = "25"
features = selection["selected"][TOP_N]

print(f"using the top-{TOP_N} list, {len(features)} feature(s):")
for name in features:
    print(" ", name)

## Build the evaluation population

One frame containing the fraud (label 1) and the held-out non-fraud (label 0).

**Deduplicate the fraud first.** A case reported by two feeds is one case. Left
un-deduplicated, the metrics overcount, and they overcount the *well-known* cases
specifically — which are the easy ones, so the metric flatters the model.

In [ ]:
CASE_KEY = manifest["case_key_column"]

before = len(fraud_labels)
deduped_cases = fraud_labels.drop_duplicates(subset=[CASE_KEY])
print(f"fraud cases: {before} rows -> {len(deduped_cases)} distinct case(s)")

fraud_customers = set(deduped_cases["customer_id"])
fraud_eval = fraud[fraud["customer_id"].isin(fraud_customers)].drop_duplicates(
    subset=["customer_id"]
).copy()
fraud_eval["label"] = 1

nonfraud_eval = testing[~testing["customer_id"].isin(fraud_customers)].copy()
nonfraud_eval["label"] = 0

evaluation = pd.concat([nonfraud_eval, fraud_eval], ignore_index=True)
evaluation = evaluation.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"evaluation population: {len(evaluation)} rows, "
      f"{int(evaluation['label'].sum())} positive ({evaluation['label'].mean():.4%})")

## Impute and scale

In [ ]:
import re

COUNT_PATTERN = re.compile(r"(_cnt$|_cnt_|_any_|_total_cnt$|_event_cnt$)")
RECENCY_PATTERN = re.compile(r"(_days_since_last|_days_since_)")
ZERO_SPREAD_PATTERN = re.compile(r"(_entropy$|_spread_days$|_over_|_frac$|_cv$|_std$|_stddev$)")


def derive_impute_rules(frame: pd.DataFrame, features: list[str]) -> dict[str, dict]:
    """One rule per feature, chosen from the column name, with the fill value
    computed from the training data only."""
    rules = {}
    for name in features:
        values = frame[name].dropna().astype(float)
        if COUNT_PATTERN.search(name):
            rules[name] = {"rule": "zero_because_never_happened", "fill": 0.0}
        elif ZERO_SPREAD_PATTERN.search(name):
            rules[name] = {"rule": "zero_because_undefined", "fill": 0.0}
        elif RECENCY_PATTERN.search(name) and "first_seen" not in name:
            sentinel = float(values.max() * 1.5) if len(values) else 9999.0
            rules[name] = {"rule": "sentinel_because_never_seen", "fill": sentinel}
        else:
            rules[name] = {
                "rule": "median",
                "fill": float(values.median()) if len(values) else 0.0,
            }
    return rules


impute_rules = derive_impute_rules(training, features)

summary = pd.DataFrame(
    [{"feature": k, **v} for k, v in impute_rules.items()]
).sort_values(["rule", "feature"])
print(summary["rule"].value_counts().to_string())
print()
display(summary.round(3))

In [ ]:
from sklearn.preprocessing import RobustScaler


def apply_impute(frame: pd.DataFrame, features: list[str], rules: dict) -> pd.DataFrame:
    out = frame[features].astype(float).copy()
    for name in features:
        out[name] = out[name].fillna(rules[name]["fill"])
    return out


train_imputed = apply_impute(training, features, impute_rules)
eval_imputed = apply_impute(evaluation, features, impute_rules)

scaler = RobustScaler().fit(train_imputed)
train_matrix = scaler.transform(train_imputed)
eval_matrix = scaler.transform(eval_imputed)

print("train matrix:", train_matrix.shape, "  any NaN:", bool(np.isnan(train_matrix).any()))
print("eval  matrix:", eval_matrix.shape, "  any NaN:", bool(np.isnan(eval_matrix).any()))

## The models

Written so that higher always means more anomalous. That normalisation is the
whole job of the adapter layer in the package, and it is easy to get backwards:
sklearn's `score_samples` returns higher for *more normal*, so it needs negating.
A sign error here produces a model that is exactly as good as the correct one at
finding the *most normal* customers, and every metric will read near zero — so if
a model scores implausibly badly, check its sign before anything else.

Optional libraries are attempted and skipped if absent. `IsolationForest`, PCA
reconstruction error, k-means distance and a Mahalanobis baseline need only
scikit-learn and will always run.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.covariance import EmpiricalCovariance
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

MODELS = {}


def register(name):
    def wrap(builder):
        MODELS[name] = builder
        return builder
    return wrap


@register("isolation_forest")
def _isolation_forest(matrix):
    model = IsolationForest(
        n_estimators=200, max_samples=min(256, len(matrix)),
        contamination="auto", random_state=SEED, n_jobs=-1,
    ).fit(matrix)
    # score_samples: higher = more normal. Negate.
    return lambda X: -model.score_samples(X), model


@register("pca_reconstruction")
def _pca(matrix):
    n = min(matrix.shape[1] - 1, max(2, int(matrix.shape[1] * 0.5)))
    model = PCA(n_components=n, random_state=SEED).fit(matrix)

    def score(X):
        return np.sum((X - model.inverse_transform(model.transform(X))) ** 2, axis=1)

    return score, model


@register("kmeans_distance")
def _kmeans(matrix):
    model = KMeans(n_clusters=8, n_init=10, random_state=SEED).fit(matrix)
    return lambda X: model.transform(X).min(axis=1), model


@register("mahalanobis")
def _mahalanobis(matrix):
    model = EmpiricalCovariance().fit(matrix)
    return lambda X: model.mahalanobis(X), model


try:
    from pyod.models.ecod import ECOD

    @register("ecod")
    def _ecod(matrix):
        model = ECOD().fit(matrix)
        return lambda X: model.decision_function(X), model

    from pyod.models.copod import COPOD

    @register("copod")
    def _copod(matrix):
        model = COPOD().fit(matrix)
        return lambda X: model.decision_function(X), model

    from pyod.models.hbos import HBOS

    @register("hbos")
    def _hbos(matrix):
        model = HBOS(n_bins=20).fit(matrix)
        return lambda X: model.decision_function(X), model
except ImportError:
    print("pyod not installed -- skipping ecod, copod, hbos")

try:
    import hdbscan

    @register("hdbscan")
    def _hdbscan(matrix):
        model = hdbscan.HDBSCAN(
            min_cluster_size=max(15, len(matrix) // 500),
            prediction_data=True,
        ).fit(matrix)

        def score(X):
            memberships = hdbscan.approximate_predict_scores(model, X)
            return 1.0 - memberships

        return score, model
except ImportError:
    print("hdbscan not installed -- skipping hdbscan")

print()
print("models to run:", sorted(MODELS))

## Fit and score

In [ ]:
import time

import joblib

MODEL_DIR = RUN_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

scores = pd.DataFrame(
    {
        "customer_id": evaluation["customer_id"].values,
        "label": evaluation["label"].values,
    }
)
fitted = {}

for name, builder in sorted(MODELS.items()):
    started = time.time()
    try:
        score_fn, model = builder(train_matrix)
        train_scores = score_fn(train_matrix)
        eval_scores = score_fn(eval_matrix)
        scores[name] = eval_scores
        fitted[name] = model
        joblib.dump(
            {
                "model": model,
                "scaler": scaler,
                "features": features,
                "impute_rules": impute_rules,
                "train_score_quantiles": np.quantile(
                    train_scores, [0.0, 0.5, 0.9, 0.99, 1.0]
                ).tolist(),
            },
            MODEL_DIR / f"{name}.joblib",
        )
        print(f"{name:20s} ok    {time.time() - started:6.1f}s  "
              f"score range {eval_scores.min():.3f} .. {eval_scores.max():.3f}")
    except Exception as exc:
        print(f"{name:20s} FAIL  {type(exc).__name__}: {exc}")

model_names = [c for c in scores.columns if c not in {"customer_id", "label"}]
print()
print("scored models:", model_names)

## Metrics that match how the model gets used

**Precision at k.** A reviewer works a queue of fixed length. `precision@500`
is: of the 500 highest-scoring customers, what fraction were fraud. This is the
number an operations lead actually cares about.

**Recall at k.** Of all the fraud, what fraction made it into the top k. The
complement of "how much did we miss".

**Lift at k.** Precision at k divided by the base rate. Lift of 20 means the
queue is twenty times denser in fraud than a random sample. This is the number
that survives a change in the base rate, so it is the one to compare across
scenarios.

**ROC-AUC** is reported for completeness and should be read with suspicion: at a
0.4% base rate it is dominated by the ranking of the negatives, which nobody
looks at.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score


def top_k_metrics(labels: np.ndarray, values: np.ndarray, k_values: list[int]) -> list[dict]:
    order = np.argsort(-values)
    ordered = labels[order]
    total_positive = int(labels.sum())
    base_rate = labels.mean()
    rows = []
    for k in k_values:
        k = min(k, len(ordered))
        hits = int(ordered[:k].sum())
        precision = hits / k
        rows.append(
            {
                "k": k,
                "hits": hits,
                "precision_at_k": precision,
                "recall_at_k": hits / max(total_positive, 1),
                "lift_at_k": precision / base_rate if base_rate > 0 else np.nan,
            }
        )
    return rows


labels = scores["label"].to_numpy()
n_rows = len(labels)
K_VALUES = sorted(
    {100, 160, 250, 500, 1000, int(n_rows * 0.005), int(n_rows * 0.01), int(n_rows * 0.05)}
    - {0}
)
print("k values:", K_VALUES)
print(f"base rate: {labels.mean():.4%}")

records = []
for name in model_names:
    values = scores[name].to_numpy()
    finite = np.isfinite(values)
    auc = roc_auc_score(labels[finite], values[finite])
    ap = average_precision_score(labels[finite], values[finite])
    for row in top_k_metrics(labels[finite], values[finite], K_VALUES):
        records.append({"model": name, "roc_auc": auc, "average_precision": ap, **row})

metrics = pd.DataFrame(records)
metrics.to_csv(RUN_DIR / "metrics.csv", index=False)

headline = (
    metrics[metrics["k"] == 500]
    .sort_values("precision_at_k", ascending=False)
    [["model", "k", "hits", "precision_at_k", "recall_at_k", "lift_at_k", "roc_auc"]]
)
display(headline.round(4))

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name in model_names:
    subset = metrics[metrics["model"] == name].sort_values("k")
    axes[0].plot(subset["k"], subset["precision_at_k"], marker="o", label=name)
    axes[1].plot(subset["k"], subset["recall_at_k"], marker="o", label=name)
    axes[2].plot(subset["k"], subset["lift_at_k"], marker="o", label=name)

axes[0].axhline(labels.mean(), linestyle="--", color="grey", label="base rate")
for axis, title in zip(axes, ["precision@k", "recall@k", "lift@k"]):
    axis.set_title(title)
    axis.set_xlabel("k (queue length)")
    axis.set_xscale("log")
axes[2].axhline(1.0, linestyle="--", color="grey")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

## Do the models agree?

If two models with different inductive biases pick the same customers, that is
corroboration. If they pick disjoint sets, an ensemble may beat either — and
either way it is a thing to know before declaring a winner.

In [ ]:
TOP_K = min(500, len(scores))
top_sets = {
    name: set(scores.nlargest(TOP_K, name)["customer_id"]) for name in model_names
}

overlap = pd.DataFrame(
    {
        left: {
            right: round(len(top_sets[left] & top_sets[right]) / TOP_K, 3)
            for right in model_names
        }
        for left in model_names
    }
)
print(f"fraction of the top-{TOP_K} shared between each pair of models:")
display(overlap)

In [ ]:
scores.to_parquet(RUN_DIR / "scores.parquet", index=False)

manifest["stages_complete"] = sorted(set(manifest["stages_complete"]) | {"03"})
manifest["paths"].update(
    {
        "scores": str(RUN_DIR / "scores.parquet"),
        "metrics": str(RUN_DIR / "metrics.csv"),
        "models": str(MODEL_DIR),
    }
)
manifest["trained"] = {
    "top_n": TOP_N,
    "features": features,
    "models": model_names,
    "k_values": K_VALUES,
    "base_rate": float(labels.mean()),
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Stage 03 complete. Open 04-compare-and-report.ipynb.")